Tunable TIA design example with the optimizer orchestrator class (SPICE-based optimization)

# Pre-body

## Clearing past runs (optional)

In [1]:
!rm -rf logs/ # clear logs
!rm -rf spice_out/

## IIC-OSIC Env Setup

In [2]:
# # The following workaround is needed to run the jupyter notebook in docker container.
import os
import subprocess

# Run the script in a shell, capture its environment
PDK = "ihp-sg13g2"
command = f"bash -c 'source /foss/tools/sak/iic-pdk-script.sh {PDK} && source ~/.bashrc && env'"
result = subprocess.run(command, capture_output=True, text=True, shell=True)

# Parse environment variables from the output
for line in result.stdout.splitlines():
    key, _, value = line.partition("=")
    if key and value:
        os.environ[key] = value
        
os.environ["PATH"] += ":/foss/tools/bin"

# Now they are in your current Python process
print("PDK_ROOT:", os.environ.get("PDK_ROOT"))
print("SPICE_USERINIT_DIR:", os.environ.get("SPICE_USERINIT_DIR"))

# Test ngspice
!ngspice -v

PDK_ROOT: /foss/pdks
SPICE_USERINIT_DIR: /foss/pdks/ihp-sg13g2/libs.tech/ngspice
******
** ngspice-44.2 : Circuit level simulation program
** Compiled with KLU Direct Linear Solver
** The U. C. Berkeley CAD Group
** Copyright 1985-1994, Regents of the University of California.
** Copyright 2001-2024, The ngspice team.
** Please get your ngspice manual from https://ngspice.sourceforge.io/docs.html
** Please file your bug-reports at http://ngspice.sourceforge.net/bugrep.html
** Creation Date: Sat May 24 09:38:33 UTC 2025
******


## Library Imports

In [3]:
import logging

from pathlib import Path

from symxplorer.optimization    import Circuit_Optimizer_Orchestrator_with_SPICE, Optimizer_Type_Enum
from symxplorer.logging         import setup_loggers

setup_loggers()
logger = logging.getLogger("SymXplorer.jupyter")
logger.info("Spicelib_Wrapper imported successfully.")

2025-10-22 07:21:18,058 - SymXplorer.base_optimizer - Using device: cpu and dtype: torch.float64
2025-10-22 07:21:18,060 - SymXplorer.Nevergrad - imported symxplorer.optimization.nevergrad
2025-10-22 07:21:18,178 - graphviz._tools - deprecate positional args: graphviz.backend.piping.pipe(['renderer', 'formatter', 'neato_no_op', 'quiet'])
2025-10-22 07:21:18,179 - graphviz._tools - deprecate positional args: graphviz.backend.rendering.render(['renderer', 'formatter', 'neato_no_op', 'quiet'])
2025-10-22 07:21:18,180 - graphviz._tools - deprecate positional args: graphviz.backend.unflattening.unflatten(['stagger', 'fanout', 'chain', 'encoding'])
2025-10-22 07:21:18,180 - graphviz._tools - deprecate positional args: graphviz.backend.viewing.view(['quiet'])
2025-10-22 07:21:18,182 - graphviz._tools - deprecate positional args: graphviz.quoting.quote(['is_html_string', 'is_valid_id', 'dot_keywords', 'endswith_odd_number_of_backslashes', 'escape_unescaped_quotes'])
2025-10-22 07:21:18,182 - g

# Instantiations


In [4]:
# ----------------------------
# Instantiations
# ----------------------------
ws_root = "/foss/designs/eda/SymXplorer/examples/tunable-tia"
pdk_name = "ihp-sg13g2"
yaml_file_name = Path("tia-topo-2/project_setup-ideal-comp.yaml")

project_setup_path = Path(f"{ws_root}/{pdk_name}/spice/{yaml_file_name}")

orchestrator = Circuit_Optimizer_Orchestrator_with_SPICE(
    project_setup_path=project_setup_path,
    optimizer_type=Optimizer_Type_Enum.NEVERGRAD_CONSTRAINT,
    verbose=False
)


07:21:18 - SymXplorer.domains: [INFO] Initialized OptimizerConfig: CMA, type=nevergrad, budget=100, random_seed=48
07:21:18 - SymXplorer.domains: [INFO] 	Linear bounds: min=0, max=100
07:21:18 - SymXplorer.domains: [INFO] 	Log bounds: min=1, max=100
07:21:18 - SymXplorer.domains: [INFO] 	Loss function: max_loss=inf, norm_method=min-max, type=mse, rescale_mag=True, include_phase_loss=False, include_mag_loss=True
07:21:18 - SymXplorer.domains: [INFO] 	Number of target specs: 2
07:21:18 - SymXplorer.domains: [INFO] 		- TargetSpec(name=fc, target=500e6, range=1.00e+08 tolerance=10000000.0, goal=exact, sim_type=ac, enable=False, error_type=relative-sigmoid, weight=10.0, enable=False, description=Center frequency)
07:21:18 - SymXplorer.domains: [INFO] 		- TargetSpec(name=gain_db, target=40, range=1.00e+02 tolerance=5, goal=exceed, sim_type=ac, enable=True, error_type=relative-sigmoid, weight=10.0, enable=True, description=gain in dB at fc)
07:21:18 - SymXplorer.domains: [INFO] Project 'Tunab

In [5]:
orchestrator.run_sanity_on_spicelib_wrapper()

07:21:18 - SymXplorer.spicelib: [INFO] 📂 Creating dedicated sanity check folder...
07:21:18 - SymXplorer.spicelib: [INFO] 🧪 Running sanity check simulation...
07:21:18 - SymXplorer.spicelib: [INFO] simulator log: /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/sizing/spice_out/sanity_check/Tunable-TIA-2_sanity.log
07:21:18 - SymXplorer.spicelib: [INFO] simulator RAW: /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/sizing/spice_out/sanity_check/Tunable-TIA-2_sanity.raw
07:21:18 - SymXplorer.spicelib: [INFO] 🔎 Verifying simulation results...
07:21:18 - SymXplorer.spicelib: [INFO] ✅ Sanity check passed 🎉


True

In [6]:
circuit_optimizer = orchestrator.get_optimizer()
type(circuit_optimizer)

07:21:18 - SymXplorer.orchestrator: [INFO] creating the circuit_optimizer of type nevergrad_constraint
07:21:18 - SymXplorer.base_optimizer: [INFO] Initialized the Nevergrad_Spice_Multi_Spec_Optimizer with 2 target specs
07:21:18 - SymXplorer.Nevergrad: [INFO] started the <class 'symxplorer.optimization.nevergrad.Nevergrad_Spice_Constraint_Satisfaction'> optimizer class
07:21:18 - SymXplorer.orchestrator: [INFO] created the circuit_optimizer; type <class 'symxplorer.optimization.nevergrad.Nevergrad_Spice_Constraint_Satisfaction'>


symxplorer.optimization.nevergrad.Nevergrad_Spice_Constraint_Satisfaction

# Main Body

## Optimization

In [7]:
circuit_optimizer.parameterize()

Dict(vbias=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_cap_size=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_ind_size=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_l=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_nfet_w=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_2_size=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}],x_dut_res_load_size=Scalar{Cl(0,100,b)}[sigma=Scalar{exp=2.03}]):{'x_dut_nfet_w': 50.0, 'x_dut_nfet_l': 50.0, 'x_dut_cap_size': 50.0, 'x_dut_res_2_size': 50.0, 'x_dut_res_load_size': 50.0, 'x_dut_ind_size': 50.0, 'vbias': 50.0}

In [8]:
circuit_optimizer.optimize()

07:21:19 - SymXplorer.base_optimizer: [INFO] Optimization process started.
07:21:19 - SymXplorer.Nevergrad: [INFO] Optimizer is set to CMA with budget = 100
Optimizing:  19%|█▉        | 19/100 [00:05<00:19,  4.15trial/s]/headless/.local/lib/python3.12/site-packages/cma/evolution_strategy.py:2936: InjectionWarning:

orphanated injected solution {'iteration': 1, 'index': 0, 'counter': 0}
                            This could be a bug in the calling order/logics or due to
                            a too small popsize used in `ask()` or when only using
                            `ask(1)` repeatedly. Please check carefully.
                            In case this is desired, the warning can be surpressed with
                            ``warnings.simplefilter("ignore", cma.evolution_strategy.InjectionWarning)``
                            

Optimizing:  24%|██▍       | 24/100 [00:06<00:18,  4.01trial/s]/headless/.local/lib/python3.12/site-packages/cma/evolution_strategy.py:2936: Injec

OptimizationLog([OptimizationLogEntry(point=OptimizationPoint(params={'x_dut_nfet_w': 4.8842146416474286e-05, 'x_dut_nfet_l': 5.917937689534287e-06, 'x_dut_cap_size': 2.0783079954356557e-11, 'x_dut_res_2_size': 5909.952766193229, 'x_dut_res_load_size': 3418.6294290864184, 'x_dut_ind_size': 4.548307706622478e-10, 'vbias': 0.5053029606855389}, score=np.float64(-1.499353834111612), metadata={}), fit_summary={'fc': {'curr_val': np.float64(1503591000.0), 'score': np.float64(-9.999031949333032)}, 'gain_db': {'curr_val': np.float64(4.785133467075496), 'score': np.float64(-1.499353834111612)}}, log_file=PosixPath('/foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/sizing/spice_out/run_2/tb_ac-ideal-comp_2.log')), OptimizationLogEntry(point=OptimizationPoint(params={'x_dut_nfet_w': 4.216466891785347e-05, 'x_dut_nfet_l': 6.902339462959453e-06, 'x_dut_cap_size': 3.1653660010343565e-11, 'x_dut_res_2_size': 4610.366384737694, 'x_dut_res_load_size': 3474.7828722385975, 'x_dut_ind_size': 4.

In [9]:
circuit_optimizer.plot_score(save_path=project_setup_path.parent / "loss_curve.html", show=True)

07:21:47 - SymXplorer.plotter: [INFO] 📊 Plot saved to /foss/designs/eda/SymXplorer/examples/tunable-tia/ihp-sg13g2/spice/tia-topo-2/loss_curve.html
07:21:47 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


## Inspection & Visualization

### (1) Best Param

In [10]:
out = circuit_optimizer.get_best_params()
if out is not None: 
    best_param, loss, metadata = out
metadata

07:21:47 - SymXplorer.base_optimizer: [INFO] best score: 0.0


{}

In [11]:
loss

0.0

In [12]:
# Print parameter sizes (convert to u)
for param in best_param:
    print(f"{param}: {best_param[param] :0.2e}")

x_dut_nfet_w: 7.13e-05
x_dut_nfet_l: 2.23e-07
x_dut_cap_size: 2.91e-11
x_dut_res_2_size: 7.28e+03
x_dut_res_load_size: 5.48e+03
x_dut_ind_size: 4.90e-10
vbias: 7.18e-01


In [13]:
circuit_optimizer.plot_solution(best_param, show_plot=True, trace_name="vout")

07:21:47 - SymXplorer.base_optimizer: [INFO] total score: 0.0
07:21:47 - SymXplorer.base_optimizer: [INFO] 	Spec 'fc': curr_val=1115604500.0, score=-9.953236562720205
07:21:47 - SymXplorer.base_optimizer: [INFO] 	Spec 'gain_db': curr_val=35.79130543750195, score=0.0


### (3) Metric Trace

In [14]:
circuit_optimizer.plot_optimization_trace(metric_x='pm', metric_y='gain_db', show=True)

07:21:48 - SymXplorer.plotter: [WARNING] metric_x 'pm' not found in optimization log


In [15]:
circuit_optimizer.plot_score_value_by_spec(spec_name="gain_db", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="fc", show=True)
circuit_optimizer.plot_score_value_by_spec(spec_name="pm", show=True)

07:21:48 - SymXplorer.plotter: [INFO] 	min score -2.1647832032804226; max score 0.0
07:21:48 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


07:21:48 - SymXplorer.plotter: [INFO] 	min score -9.99989014708454; max score -1.9463240826008676
07:21:48 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


07:21:48 - SymXplorer.plotter: [WARNING] spec_name 'pm' not found in optimization log


### (4) Design Space Exploration

In [16]:
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_nfet_w", param_y="x_dut_nfet_l", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="x_dut_res_2_size", param_y="x_dut_res_load_size", show=True)
circuit_optimizer.plot_design_space_exploration(param_x="vbias", param_y="x_dut_ind_size", show=True)

07:21:48 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


07:21:48 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


07:21:48 - SymXplorer.plotter: [INFO] Opening interactive plot in browser...


(tensor([0.5053, 0.6668, 0.5651, 0.4537, 0.6414, 0.5075, 0.4781, 0.6223, 0.5502,
         0.5248, 0.5492, 0.3414, 0.4143, 0.4597, 0.4826, 0.4958, 0.2577, 0.4726,
         0.4742, 0.4940, 0.5447, 0.4996, 0.5432, 0.3954, 0.2514, 0.3018, 0.4591,
         0.1993, 0.2773, 0.1450, 0.1347, 0.5354, 0.2472, 0.4510, 0.5368, 0.4900,
         0.4662, 0.5618, 0.4103, 0.6511, 0.3232, 0.4290, 0.4894, 0.8051, 0.4602,
         0.8186, 0.5862, 0.6260, 0.6065, 0.5694, 0.7089, 0.3903, 0.6857, 0.5023,
         0.7220, 0.8440, 0.7183, 0.7825, 0.8729, 0.6935, 0.6246, 0.7899, 0.5318,
         0.7441, 0.8852, 0.6036, 0.8680, 0.7904, 0.4432, 0.9284, 0.7001, 0.7555,
         0.9123, 0.8761, 0.8675, 0.9521, 0.8499, 1.0043, 0.9136, 0.8329, 0.7681,
         0.7830, 0.6543, 0.7335, 0.6599, 0.8173, 0.7040, 0.6890, 0.6584, 0.7879,
         0.7495, 0.7054, 0.7032, 0.7290, 0.6548, 0.7233, 0.7738, 0.6767, 0.6948,
         0.7275]),
 tensor([4.5483e-10, 4.7376e-10, 6.6547e-10, 4.3350e-10, 5.7277e-10, 3.9216e-10,
         